# H&M seed 43 M1·현재 M2 학습예산 진단

- M1과 개인이력 `q_N/q_V` M2를 같은 seed 43으로 300 epoch 학습합니다.
- 전체 validation 평가는 100·200·300 epoch에서만 수행합니다.
- 매 epoch Drive에 optimizer·난수상태를 저장하므로 런타임 중단 후 같은 노트북을 다시 실행하면 이어집니다.
- test와 holdout은 만들거나 평가하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path('/content/clv-m2-lightgcn-runner')
SOURCE_COMMIT = '0d5bb84'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('source commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import json
import lightgcn_clv_m2_training_budget_hm2y as budget

cfg = budget.configure_hm2y_training_budget()
summary = budget.preflight_summary(cfg)
assert summary['seed'] == 43
assert summary['evaluated_at_epochs'] == [100, 200, 300]
assert summary['fixed']['test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
curve = budget.run_hm2y_training_budget(cfg)


In [ ]:
from IPython.display import display

print('1) 100·200·300 epoch 전체 절대지표')
display(curve)
print('2) 같은 epoch의 M2−M1 전체 비교')
display(curve.attrs['gap'])
print('3) 탐색적 최고점 비교 — epoch 선택 근거가 아님')
print(json.dumps(curve.attrs['exploratory_peak_comparison'], ensure_ascii=False, indent=2))
print('결과 파일:', curve.attrs['result_paths'])
